In [2]:
import pandas as pd
df = pd.read_csv("Train_original.csv")
df["label"].unique()


array(['sadness', 'neutral', 'fear', 'happiness', 'anger', 'disgust',
       'surprize', nan], dtype=object)

In [3]:
df["label"] = df["label"].str.lower()

df["label"] = df["label"].replace({"surprize": "surprise"})

df = df.dropna(subset=["label"])

print(df["label"].unique())

['sadness' 'neutral' 'fear' 'happiness' 'anger' 'disgust' 'surprise']


In [4]:
df.head()

,text,label,POS_Tags,POS_Encoded
0,خیلی ناراحت شدم وقتی خبر بدی شنیدم 236.,sadness,"['ADV', 'ADJ', 'VERB', 'NOUN', 'NOUN', 'ADJ', ...","[2, 0, 14, 7, 7, 0, 14, 7, 12]"
1,یک فرشته به خواب یکنفر میاد و بهش میگه خدا گفت...,neutral,"['NUM', 'NOUN', 'ADP', 'NOUN', 'NOUN', 'VERB',...","[8, 7, 1, 7, 7, 14, 4, 1, 10, 14, 7, 14, 7, 14..."
2,ترسیدم چون صدای عجیبی شنیدم 12201.,fear,"['VERB', 'SCONJ', 'NOUN', 'ADJ', 'VERB', 'NOUN...","[14, 13, 7, 0, 14, 7, 12]"
3,اونقدر حواسمون بود که بقیه ناراحت نشن که الان ...,sadness,"['NOUN', 'ADJ', 'AUX', 'SCONJ', 'NOUN', 'ADJ',...","[7, 0, 3, 13, 7, 0, 14, 13, 7, 10, 7, 14, 1, 8..."
4,خیلی راحته که ... کدوم جالش..,neutral,"['ADV', 'ADJ', 'SCONJ', 'PUNCT', 'PUNCT', 'PUN...","[2, 0, 13, 12, 12, 12, 7, 7, 10, 12, 12]"


In [5]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

label_mapping = {
    "happiness": 0,
    "sadness": 1,
    "surprise": 2,
    "anger": 3,
    "disgust": 4,
    "fear": 5,
    "neutral": 6
}

df["label_id"] = df["label"].map(label_mapping)

df = df.dropna(subset=["label_id"])

df["label_id"] = df["label_id"].astype(int)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label_id"]),
    y=df["label_id"]
)

# Convert to dictionary for Keras
class_weights = {i: weight for i, weight in enumerate(class_weights)}

print("Class weights:", class_weights)


Class weights: {0: 1.2886753246753246, 1: 1.023644466452092, 2: 1.3611896073966363, 3: 0.8038691488844603, 4: 1.2634071810542398, 5: 1.1824681824681824, 6: 0.613018014678627}


In [6]:
from sklearn.model_selection import train_test_split

# Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label_id"],
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

# Tokenization
from transformers import AutoTokenizer

model_name = "HooshvareLab/bert-base-parsbert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)


In [8]:
import tensorflow as tf

# Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).shuffle(1000).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_labels
)).batch(16)


2025-10-21 21:10:14.002198: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-21 21:10:14.030219: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-21 21:10:14.039139: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-21 21:10:14.061784: I tensorflow/core/platform/cpu_feature_guard.cc:211] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-21 21:10:23.192429: W tensorflo

In [10]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=7
)


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at HooshvareLab/bert-base-parsbert-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
from transformers import create_optimizer

batch_size = 16
num_epochs = 5
batches_per_epoch = len(train_texts) // batch_size
total_train_steps = batches_per_epoch * num_epochs

optimizer, lr_schedule = create_optimizer(
    init_lr=2e-5,
    num_train_steps=total_train_steps,
    num_warmup_steps=int(0.1 * total_train_steps),
)


''+ptx85' is not a recognized feature for this target' (ignoring feature)
+ptx85+ptx85' is not a recognized feature for this target (ignoring feature)
' is not a recognized feature for this target'+ptx85 (ignoring feature)
' is not a recognized feature for this target' (ignoring feature)
+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
''+ptx85' is not a recognized feature for this target (ignoring feature)
+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

In [12]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy

model.compile(
    optimizer=optimizer,
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=[SparseCategoricalAccuracy("accuracy")]
)


In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=5,
    class_weight=class_weights,
    callbacks=[early_stopping]
)


Epoch 1/5


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
''+ptx85+ptx85' is not a recognized feature for this target' is not a recognized feature for this target (ignoring feature)
 (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

   1/2481 [..............................] - ETA: 47:11:43 - loss: 1.7925 - accuracy: 0.0625

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


   2/2481 [..............................] - ETA: 23:16 - loss: 1.7840 - accuracy: 0.1250   

...

2120/2481 [========================>.....] - ETA: 1:05 - loss: 0.8669 - accuracy: 0.6598

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


2301/2481 [==========================>...] - ETA: 32s - loss: 0.8611 - accuracy: 0.6615 

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


2481/2481 [==============================] - 573s 231ms/step - loss: 0.8571 - accuracy: 0.6629 - val_loss: 1.5051 - val_accuracy: 0.4795


In [14]:
import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score

preds = model.predict(test_dataset).logits
y_pred = np.argmax(preds, axis=1)

accuracy = accuracy_score(test_labels, y_pred)
f1 = f1_score(test_labels, y_pred, average='weighted')

print("Accuracy:", round(accuracy, 3))
print("Weighted F1-score:", round(f1, 3))
print("\nClassification Report:\n")
print(classification_report(test_labels, y_pred, digits=3))


621/621 [==============================] - 124s 193ms/step
Accuracy: 0.536
Weighted F1-score: 0.541

Classification Report:

              precision    recall  f1-score   support

           0      0.727     0.521     0.607      1100
           1      0.777     0.410     0.537      1385
           2      0.578     0.600     0.589      1041
           3      0.705     0.395     0.506      1763
           4      0.759     0.447     0.563      1122
           5      0.546     0.587     0.566      1199
           6      0.377     0.715     0.494      2313

    accuracy                          0.536      9923
   macro avg      0.639     0.525     0.552      9923
weighted avg      0.615     0.536     0.541      9923



In [15]:
import pandas as pd

df_test = pd.read_csv("Test_original.csv")

label_mapping = {
    "happiness": 0,
    "sadness": 1,
    "surprise": 2,
    "anger": 3,
    "disgust": 4,
    "fear": 5,
    "neutral": 6
}

df_test["label_id"] = df_test["label"].map(label_mapping)

df_test = df_test.dropna(subset=["label_id"])
df_test["label_id"] = df_test["label_id"].astype(int)

external_test_texts = df_test["text"].astype(str).tolist()
external_test_labels = df_test["label_id"].astype(int).tolist()

external_encodings = tokenizer(
    external_test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

external_dataset = tf.data.Dataset.from_tensor_slices((
    dict(external_encodings),
    external_test_labels
)).batch(16)

# Predict
external_preds = model.predict(external_dataset).logits
external_y_pred = np.argmax(external_preds, axis=1)

# Evaluate
external_accuracy = accuracy_score(external_test_labels, external_y_pred)
external_f1 = f1_score(external_test_labels, external_y_pred, average='weighted')

print(" External Test_original.csv Evaluation:")
print("Accuracy:", round(external_accuracy, 3))
print("Weighted F1-score:", round(external_f1, 3))
print("\nClassification Report:")
print(classification_report(external_test_labels, external_y_pred, digits=3))


222/222 [==============================] - 60s 172ms/step
 External Test_original.csv Evaluation:
Accuracy: 0.432
Weighted F1-score: 0.361

Classification Report:
              precision    recall  f1-score   support

           0      0.435     0.512     0.471       547
           1      1.000     0.011     0.022       538
           2      0.234     0.585     0.334       183
           3      0.429     0.022     0.041       414
           4      0.000     0.000     0.000        56
           5      0.360     0.127     0.188       212
           6      0.471     0.691     0.560      1600

    accuracy                          0.432      3550
   macro avg      0.418     0.278     0.231      3550
weighted avg      0.514     0.432     0.361      3550



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
